In [ ]:
from google import genai

client = genai.Client()

batches = {
    "gemini-batch-001": "batches/ru4wk8l260vodttosor0xeyc5sv2zt34k3yt",
    "gemini-batch-002": "batches/e1wgn18irescbplbgwuc5jhfewwyxtwp8wdh",
    "gemini-batch-003": "batches/6msp378h4ogw4dkm0f6juquveeqyp4eg5i0n",
    "gemini-batch-004": "batches/77yvuog4uy34ezhf6hzh6q4u88p95u6a511w",
}

for id, job_name in batches.items():
    
    batch_job = client.batches.get(name=job_name)

    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("Downloading result file content...")
    file_content = client.files.download(file=result_file_name)

    file_path = f"results/{id}.jsonl"
    with open(file_path, "w") as f:
        f.write(file_content.decode('utf-8'))
        print(f"wrote file {file_path}")

In [ ]:
from openai import OpenAI

client = OpenAI()

batches = {
    "openai-batch-001": "batch_69dbb2671a5081909e1af4c27727202d",
    "openai-batch-002": "batch_69dbb794b61c81909ed827f2b85d84a1",
    "openai-batch-003": "batch_69dbbb28d08881909a60eae12c4356c3",
    "openai-batch-004": "batch_69dbc13b8ac48190957e243f4bf49df0",
}

for batch_name, batch_id in batches.items():
    batch = client.batches.retrieve(batch_id)
    output_file_id = batch.output_file_id

    print(f"Results are in file: {output_file_id}")
    print("Downloading result file content...")

    file_content = client.files.content(output_file_id).text
    output_path = f"results{batch_name}.jsonl"
    output_path.write_text(file_content, encoding="utf-8")
    print(f"wrote file {output_path}")

In [ ]:
from anthropic import Anthropic

client = Anthropic()

batches = {
    "anthropic-batch-001": ["msgbatch_018TjDKWNk9Z5SXLsNTZD8F6"],
    "anthropic-batch-002": ["msgbatch_01CnUpQLbvq4qWFbMJJV3pyf"],
    "anthropic-batch-003": ["msgbatch_01VUB2i1fazY3VBo66UWBodW", "msgbatch_01JtBSZP9v84h3mh9EUos25C"],
    "anthropic-batch-004": ["msgbatch_01VM6KgNzqqj6oYW37LJDdQj"],
}

for batch_name, batch_ids in batches.items():
    output_path = f"results/{batch_name}.jsonl"
    batch_result_lines: list[list[str]] = []

    for batch_id in batch_ids:
        batch = client.messages.batches.retrieve(batch_id)
        print(f"Batch {batch_id} status: {batch.processing_status}")
        print(f"Downloading results for: {batch_id}")
        current_batch_lines = [
            entry.model_dump_json() for entry in client.messages.batches.results(batch_id)
        ]
        batch_result_lines.append(current_batch_lines)

    if batch_name == "anthropic-batch-003":
        retry_lines = batch_result_lines[1]
        retry_count = len(retry_lines)
        first_batch_lines = batch_result_lines[0]
        result_lines = first_batch_lines[:-retry_count] + retry_lines
    else:
        result_lines = [line for lines in batch_result_lines for line in lines]

    output_path.write_text("\n".join(result_lines) + "\n", encoding="utf-8")
    print(f"wrote file {output_path}")